In [ ]:
"""
MULTI-TASK — MIXED REGRESSION + CLASSIFICATION
  force   : regression  → MSE loss  → MAE/RMSE metrics
  angle   : regression  → MSE loss  → MAE/RMSE metrics
  bending : regression  → MSE loss  → MAE/RMSE metrics
  height  : classification → CrossEntropy loss → Accuracy/F1 metrics

Change NUM_HEIGHT_CLASSES to match your actual number of height levels.
"""

# ============================================================================
# CELL 1: SETUP
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations as _combinations
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              accuracy_score, f1_score, confusion_matrix)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from scipy import stats

print("Setup complete!")
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

# ============================================================================
# CELL 2: CONFIG
# ============================================================================

DATA_FILES = [
    # synced: force + height + bending tasks
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_1_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_2_rotate_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_3_rotate_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_4_height_processed.csv",
    # synced2: joint angle prediction (separate recording session)
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced2/group_2_rotate_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced2/group_3_rotate_processed.csv",
]

BASE_SAVE_DIR    = "/content/drive/MyDrive/Multi_Modal_Rehab/multitask_mixed"
SEQ_LENGTH       = 30
STRIDE           = 2
BATCH_SIZE       = 32
NUM_EPOCHS       = 200
PATIENCE         = 15
LEARNING_RATE    = 1e-3

# ============================================================================
# CELL 3: TASK CONFIG + MODEL CONFIG
# ============================================================================

# ── EDIT THIS: set to your actual number of height levels ──
NUM_HEIGHT_CLASSES = 8   # 8 height levels

# TASK_CONFIG: name → (column_candidates, output_dim, loss_weight, task_type)
#   task_type: "regression" | "classification"
#   output_dim for classification = num_classes
TASK_CONFIG = {
    "force":       (["adc_avg", "force", "force_n", "adc_mean"],           1,                  1.0, "regression"),
    "angle":       (["angle", "angle_deg", "joint_angle_degree",
                     "angle_degree", "theta"],                             1,                  1.0, "regression"),
    "bending":     (["angle", "angle_deg", "bend", "bend_angle"],   1,                  1.0, "regression"),
    "height":      (["height", "touch_height", "height_mm",
                     "height_level", "height_label", "level"],            NUM_HEIGHT_CLASSES,  1.0, "classification"),
    # 5th task: joint angle from separate synced2 rotate session
    "joint_angle": (["joint_angle_degree",
                     "angle_degree", "theta"],                             1,                  1.0, "regression"),
}

V2 = {
    'eit': dict(input_size=8,  hidden_size=128, num_layers=3),
    'emg': dict(input_size=4,  hidden_size=64,  num_layers=2),
    'cap': dict(input_size=1,  hidden_size=32,  num_layers=1),
}

ALL_COMBOS = [
    list(c)
    for r in [1, 2, 3]
    for c in _combinations(['eit', 'emg', 'cap'], r)
]

SHARED_SIZE    = 256
DECODER_HIDDEN = 128
DECODER_LAYERS = 2
DROPOUT        = 0.3

# ── FILE-TASK RESTRICTIONS ──────────────────────────────────────────────────
# Maps source_file keywords to which tasks are ACTIVE in that file.
# Tasks not listed for a file will be masked out (loss ignored).
# None = no restriction (task active in all files if column exists).
FILE_TASK_MAP = {
    "group_1":        ["force", "angle", "bending"],   # synced group1
    "group_2_rotate": ["angle", "bending"],            # synced group2
    "group_3_rotate": ["angle", "bending"],            # synced group3
    "group_4":        ["height"],                      # synced group4: height only
    "synced2":        ["joint_angle"],                 # synced2: 5th task only
}

print(f"Tasks  : {list(TASK_CONFIG.keys())}  (5 tasks)")
print(f"Height : {NUM_HEIGHT_CLASSES}-class classification")
print(f"Joint angle: from synced2 rotate files, masked in other files")
print(f"Combos : {['+'.join(c) for c in ALL_COMBOS]}")

# ============================================================================
# CELL 4: HELPERS
# ============================================================================

# Force synchronous CUDA execution so errors point to real location
import os as _os
_os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def safe_pearsonr(a, b):
    a, b = np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)
    if a.size < 5 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(stats.pearsonr(a, b)[0])

def find_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def load_and_merge(file_list):
    dfs = []
    for fp in file_list:
        if not os.path.exists(fp):
            print(f"  Not found, skipping: {fp}")
            continue
        df = pd.read_csv(fp)
        df["source_file"] = os.path.basename(fp)
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    print(f"Merged: {merged.shape}  files: {merged['source_file'].unique().tolist()}")
    return merged

# ============================================================================
# CELL 5: DATASET
# ============================================================================

class MultiTaskDataset(Dataset):
    """
    Regression targets   : stored as [T, 1] float
    Classification target: stored as [T]    long  (class indices 0..C-1)
    Masks                : [T, 1] float, 1=valid 0=missing
    """
    def __init__(self, df, seq_length=30, stride=2, fill_value=0.0):
        self.seq_length = int(seq_length)
        self.stride     = int(stride)
        df = df.copy()

        eit_cols = [f"eit_ch{i}" for i in range(1, 9)]
        emg_cols = [f"emg{i}" for i in range(4)]
        cap_cols = ["cap0_ma"]

        self.task_cols = {}
        for task, (candidates, _, _, _) in TASK_CONFIG.items():
            col = find_col(df, candidates)
            if col:
                self.task_cols[task] = col
                print(f"  Task '{task}' -> '{col}'")
            else:
                print(f"  Task '{task}' -> not found, will be masked")


        all_sensor = eit_cols + emg_cols + cap_cols
        for c in all_sensor:
            if c not in df.columns:
                raise ValueError(f"Missing: {c}")

        if "time_sec" in df.columns:
            keys = ["source_file"] + (["segment_id"] if "segment_id" in df.columns else []) + ["time_sec"]
            df = df.sort_values(keys).reset_index(drop=True)

        for c in all_sensor + list(self.task_cols.values()):
            df[c] = pd.to_numeric(df[c], errors="coerce")

        eit = df[eit_cols].to_numpy(np.float32)
        emg = df[emg_cols].to_numpy(np.float32)
        cap = df[cap_cols].to_numpy(np.float32)
        task_arrays = {t: df[[col]].to_numpy(np.float32)
                       for t, col in self.task_cols.items()}

        valid = (np.isfinite(eit).any(1) | np.isfinite(emg).any(1) | np.isfinite(cap).any(1))
        eit, emg, cap = eit[valid], emg[valid], cap[valid]
        for t in task_arrays:
            task_arrays[t] = task_arrays[t][valid]

        def clean(arr):
            arr[np.isinf(arr)] = np.nan
            return np.nan_to_num(arr, nan=fill_value,
                                 posinf=fill_value, neginf=fill_value).astype(np.float32)

        eit, emg, cap = clean(eit), clean(emg), clean(cap)
        src = df["source_file"].values[valid]

        self.sequences = []
        for fname in np.unique(src):
            idx = np.where(src == fname)[0]
            n   = len(idx)
            if n < self.seq_length:
                continue
            eit_f  = eit[idx]; emg_f = emg[idx]; cap_f = cap[idx]
            task_f = {t: task_arrays[t][idx] for t in task_arrays}

            # determine which tasks are active for this source file
            # using FILE_TASK_MAP keyword matching
            active_tasks_for_file = set()
            for keyword, allowed in FILE_TASK_MAP.items():
                if keyword in fname:
                    active_tasks_for_file.update(allowed)
            # if no keyword matched, allow all tasks that have a column
            if not active_tasks_for_file:
                active_tasks_for_file = set(task_f.keys())

            for i in range((n - self.seq_length) // self.stride + 1):
                s, e = i * self.stride, i * self.stride + self.seq_length
                if not any(np.isfinite(task_f[t][s:e]).any() for t in task_f):
                    continue

                window = {"eit": eit_f[s:e], "emg": emg_f[s:e], "cap": cap_f[s:e]}

                for task, (_, out_dim, _, task_type) in TASK_CONFIG.items():
                    # task active only if column exists AND file allows this task
                    task_active = (task in task_f) and (task in active_tasks_for_file)

                    if task_active:
                        tw  = task_f[task][s:e].copy()
                        msk = np.isfinite(tw).astype(np.float32)
                        if task_type == "classification":
                            n_classes = TASK_CONFIG[task][1]
                            tw_int = np.where(
                                np.isfinite(tw),
                                tw.round().astype(np.int64),
                                -1
                            ).reshape(-1)
                            valid = tw_int >= 0
                            if valid.any():
                                unique_vals = np.unique(tw_int[valid])
                                # always remap to 0-indexed (handles 1-based, gap labels, etc.)
                                val_to_idx = {v: i for i, v in enumerate(unique_vals)}
                                remapped = np.array([val_to_idx[v] if v in val_to_idx else -1
                                                     for v in tw_int])
                                tw_int = np.where(valid, remapped, -1).astype(np.int64)
                                # final safety clamp
                                tw_int = np.where(tw_int >= 0,
                                                  np.clip(tw_int, 0, n_classes - 1),
                                                  -1).astype(np.int64)
                            window[f"target_{task}"] = tw_int
                        else:
                            tw = np.nan_to_num(tw, nan=0.0).astype(np.float32)
                            window[f"target_{task}"] = tw
                    else:
                        # not available in this file → zero target + zero mask
                        msk = np.zeros((self.seq_length, 1), dtype=np.float32)
                        if task_type == "classification":
                            window[f"target_{task}"] = np.full(
                                self.seq_length, -1, dtype=np.int64)
                        else:
                            window[f"target_{task}"] = np.zeros(
                                (self.seq_length, 1), dtype=np.float32)

                    window[f"mask_{task}"] = msk

                self.sequences.append(window)

        print(f"  Total windows: {len(self.sequences)}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        out = {}
        for k, v in s.items():
            if "target_height" in k and TASK_CONFIG["height"][3] == "classification":
                out[k] = torch.tensor(v, dtype=torch.long)
            else:
                out[k] = torch.tensor(v, dtype=torch.float32)
        return out

# ============================================================================
# CELL 6: MODEL
# ============================================================================

class ModalityEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return h[-1]

class FusionLayer(nn.Module):
    def __init__(self, fusion_input_size, shared_size, dropout=0.3):
        super().__init__()
        self.fc      = nn.Linear(fusion_input_size, shared_size)
        self.bn      = nn.BatchNorm1d(shared_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, feats):
        x = torch.cat(feats, dim=-1) if len(feats) > 1 else feats[0]
        return self.dropout(torch.relu(self.bn(self.fc(x))))

class TaskDecoder(nn.Module):
    """Shared decoder structure; output_size=1 for regression, C for classification."""
    def __init__(self, input_size, hidden_size, output_size, seq_length,
                 num_layers=2, dropout=0.2):
        super().__init__()
        self.seq_length = seq_length
        self.fc_in  = nn.Linear(input_size, hidden_size)
        self.lstm   = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True,
                              dropout=dropout if num_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hidden_size, output_size)

    def forward(self, shared):
        x = torch.relu(self.fc_in(shared))
        x = x.unsqueeze(1).repeat(1, self.seq_length, 1)
        out, _ = self.lstm(x)
        return self.fc_out(out)   # [B, T, output_size]


class MultiTaskV2Model(nn.Module):
    def __init__(self, input_modalities, seq_length,
                 shared_size=256, decoder_hidden=128,
                 decoder_layers=2, dropout=0.3):
        super().__init__()
        self.input_modalities = list(input_modalities)

        self.encoders = nn.ModuleDict({
            mod: ModalityEncoder(V2[mod]['input_size'], V2[mod]['hidden_size'],
                                 V2[mod]['num_layers'], dropout)
            for mod in self.input_modalities
        })

        fusion_in   = sum(V2[m]['hidden_size'] for m in self.input_modalities)
        self.fusion = FusionLayer(fusion_in, shared_size, dropout)

        # each decoder head uses the correct output_dim from TASK_CONFIG
        self.decoders = nn.ModuleDict({
            task: TaskDecoder(
                input_size   = shared_size,
                hidden_size  = decoder_hidden,
                output_size  = TASK_CONFIG[task][1],   # 1 for reg, C for cls
                seq_length   = seq_length,
                num_layers   = decoder_layers,
                dropout      = dropout
            )
            for task in TASK_CONFIG
        })

    def forward(self, batch):
        feats  = [self.encoders[m](batch[m]) for m in self.input_modalities]
        shared = self.fusion(feats)
        return {task: self.decoders[task](shared) for task in self.decoders}

# ============================================================================
# CELL 7: MIXED LOSS
# ============================================================================

def multitask_loss(preds, batch, device):
    mse_crit = nn.MSELoss(reduction='none')
    ce_crit  = nn.CrossEntropyLoss(ignore_index=-1)  # -1 = masked/missing

    total_loss  = torch.tensor(0.0, device=device)
    task_losses = {}

    for task, (_, out_dim, weight, task_type) in TASK_CONFIG.items():
        pred = preds[task]   # [B, T, out_dim]
        mask = batch[f"mask_{task}"].to(device)   # [B, T, 1]

        if task_type == "regression":
            target = batch[f"target_{task}"].to(device)   # [B, T, 1]
            if mask.sum() < 1:
                task_losses[task] = 0.0; continue
            ml = (mse_crit(pred, target) * mask).sum() / mask.sum()

        else:  # classification
            # pred: [B, T, C]  target: [B, T] long  (-1 = ignore)
            target = batch[f"target_{task}"].to(device)   # [B, T]
            B, T, C = pred.shape
            # CrossEntropyLoss expects [N, C] and [N]
            ml = ce_crit(pred.reshape(B * T, C),
                         target.reshape(B * T))

        if not torch.isfinite(ml):
            task_losses[task] = 0.0; continue

        total_loss = total_loss + weight * ml
        task_losses[task] = ml.item()

    return total_loss, task_losses

# ============================================================================
# CELL 8: TRAIN / VAL / EVAL
# ============================================================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    totals = []; task_sums = {t: [] for t in TASK_CONFIG}
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, tl = multitask_loss(model(batch), batch, device)
        if not torch.isfinite(loss): continue
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        totals.append(loss.item())
        for t, v in tl.items():
            if v > 0: task_sums[t].append(v)
    avg   = float(np.mean(totals)) if totals else 0.0
    avg_t = {t: float(np.mean(v)) if v else 0.0 for t, v in task_sums.items()}
    return avg, avg_t

@torch.no_grad()
def validate_epoch(model, loader, device):
    model.eval()
    losses = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, _ = multitask_loss(model(batch), batch, device)
        if torch.isfinite(loss): losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    reg_p  = {t: [] for t in TASK_CONFIG if TASK_CONFIG[t][3] == "regression"}
    reg_g  = {t: [] for t in TASK_CONFIG if TASK_CONFIG[t][3] == "regression"}
    cls_p  = {t: [] for t in TASK_CONFIG if TASK_CONFIG[t][3] == "classification"}
    cls_g  = {t: [] for t in TASK_CONFIG if TASK_CONFIG[t][3] == "classification"}

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model(batch)

        for task, (_, out_dim, _, task_type) in TASK_CONFIG.items():
            pred = preds[task]   # [B, T, out_dim]

            if task_type == "regression":
                mask = batch[f"mask_{task}"].cpu().numpy().reshape(-1) > 0
                reg_p[task].append(pred.cpu().numpy().reshape(-1)[mask])
                reg_g[task].append(batch[f"target_{task}"].cpu().numpy().reshape(-1)[mask])

            else:  # classification
                target = batch[f"target_{task}"].cpu().numpy().reshape(-1)  # [B*T]
                # argmax over class dim
                pred_cls = pred.argmax(dim=-1).cpu().numpy().reshape(-1)    # [B*T]
                # only keep valid (not -1)
                valid = target >= 0
                cls_p[task].append(pred_cls[valid])
                cls_g[task].append(target[valid])

    metrics = {}

    # regression metrics
    for t in reg_p:
        if not reg_p[t] or sum(len(a) for a in reg_p[t]) == 0:
            metrics[t] = None; continue
        p = np.concatenate(reg_p[t]); g = np.concatenate(reg_g[t])
        mse    = mean_squared_error(g, p)
        ss_res = np.sum((g - p) ** 2)
        ss_tot = np.sum((g - g.mean()) ** 2)
        metrics[t] = {
            "type": "regression",
            "mae":  float(mean_absolute_error(g, p)),
            "rmse": float(np.sqrt(mse)),
            "r2":   float(1 - ss_res / ss_tot) if ss_tot > 1e-12 else 0.0,
            "corr": safe_pearsonr(p, g),
        }

    # classification metrics
    for t in cls_p:
        if not cls_p[t] or sum(len(a) for a in cls_p[t]) == 0:
            metrics[t] = None; continue
        p = np.concatenate(cls_p[t]); g = np.concatenate(cls_g[t])
        acc  = float(accuracy_score(g, p))
        f1_w = float(f1_score(g, p, average='weighted', zero_division=0))
        f1_m = float(f1_score(g, p, average='macro',    zero_division=0))
        cm   = confusion_matrix(g, p)
        metrics[t] = {
            "type":     "classification",
            "accuracy": acc,
            "f1_weighted": f1_w,
            "f1_macro":    f1_m,
            "confusion_matrix": cm.tolist(),
        }

    return metrics

# ============================================================================
# CELL 9: MAIN — loop all 7 combos
# ============================================================================

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(BASE_SAVE_DIR, exist_ok=True)
print(f"\nDevice: {device}")

df_merged = load_and_merge(DATA_FILES)
print("\nBuilding dataset...")
full_ds = MultiTaskDataset(df_merged, seq_length=SEQ_LENGTH, stride=STRIDE)

n       = len(full_ds)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Windows -> train:{len(train_ds)}  val:{len(val_ds)}  test:{len(test_ds)}\n")

all_results = []

for combo in ALL_COMBOS:
    combo_str = '+'.join(combo)
    fusion_in = sum(V2[m]['hidden_size'] for m in combo)

    print(f"\n{'='*60}")
    print(f"COMBO: {combo_str}  (fusion_in={fusion_in})")
    print(f"{'='*60}")

    save_dir  = os.path.join(BASE_SAVE_DIR, combo_str)
    os.makedirs(save_dir, exist_ok=True)

    model = MultiTaskV2Model(
        input_modalities = combo,
        seq_length       = SEQ_LENGTH,
        shared_size      = SHARED_SIZE,
        decoder_hidden   = DECODER_HIDDEN,
        decoder_layers   = DECODER_LAYERS,
        dropout          = DROPOUT,
    ).to(device)

    n_params  = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,}")

    optimizer  = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
    best_val   = float("inf")
    no_improve = 0
    ckpt_path  = os.path.join(save_dir, "best_model.pth")

    for epoch in range(NUM_EPOCHS):
        tr, tr_t = train_epoch(model, train_loader, optimizer, device)
        va        = validate_epoch(model, val_loader, device)
        scheduler.step(va)

        if va < best_val:
            best_val, no_improve = va, 0
            torch.save({"epoch": epoch, "model": model.state_dict()}, ckpt_path)
        else:
            no_improve += 1

        if (epoch + 1) % 20 == 0 or no_improve == PATIENCE:
            task_str = "  ".join(f"{t}={tr_t.get(t,0):.4f}" for t in TASK_CONFIG)
            print(f"  Ep {epoch+1:>3}  total={tr:.5f}  val={va:.5f}  | {task_str}"
                  + (" <- best" if no_improve == 0 else f"  (no_imp {no_improve}/{PATIENCE})"))

        if no_improve >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}"); break

    print(f"  Best val: {best_val:.6f}")
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["model"])
    metrics = evaluate(model, test_loader, device)

    # print per-task results
    print(f"\n  Results:")
    for t, m in metrics.items():
        if m is None:
            print(f"  {t:<10}  (no data)")
        elif m["type"] == "regression":
            print(f"  {t:<10} MAE={m['mae']:.4f}  RMSE={m['rmse']:.4f}  "
                  f"R2={m['r2']:.4f}  Corr={m['corr']:.4f}")
        else:
            print(f"  {t:<10} Acc={m['accuracy']:.4f}  "
                  f"F1_w={m['f1_weighted']:.4f}  F1_m={m['f1_macro']:.4f}")
            print(f"  {'':10} Confusion matrix:\n{np.array(m['confusion_matrix'])}")

    all_results.append({"combo": combo_str, "params": n_params, "metrics": metrics})

# ============================================================================
# CELL 10: FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 75)
print("FINAL SUMMARY — MIXED MULTI-TASK  (5 tasks)")
print("=" * 75)

# regression tasks
for task in [t for t in TASK_CONFIG if TASK_CONFIG[t][3] == "regression"]:
    print(f"\n  {task.upper()} (regression)")
    print(f"  {'Combo':<20} {'MAE':>8}  {'RMSE':>8}  {'R2':>7}  {'Corr':>7}")
    print("  " + "-" * 52)
    rows = [(r["combo"], r["metrics"].get(task)) for r in all_results]
    rows.sort(key=lambda x: x[1]["mae"] if x[1] else 999)
    for combo_str, m in rows:
        if m:
            print(f"  {combo_str:<20} {m['mae']:>8.4f}  {m['rmse']:>8.4f}  "
                  f"{m['r2']:>7.4f}  {m['corr']:>7.4f}")

# classification tasks
for task in [t for t in TASK_CONFIG if TASK_CONFIG[t][3] == "classification"]:
    print(f"\n  {task.upper()} (classification, {NUM_HEIGHT_CLASSES} classes)")
    print(f"  {'Combo':<20} {'Accuracy':>10}  {'F1_weighted':>12}  {'F1_macro':>10}")
    print("  " + "-" * 58)
    rows = [(r["combo"], r["metrics"].get(task)) for r in all_results]
    rows.sort(key=lambda x: -x[1]["accuracy"] if x[1] else -999)
    for combo_str, m in rows:
        if m:
            print(f"  {combo_str:<20} {m['accuracy']:>10.4f}  "
                  f"{m['f1_weighted']:>12.4f}  {m['f1_macro']:>10.4f}")

# save CSV
rows_flat = []
for r in all_results:
    for t, m in r["metrics"].items():
        if m:
            row = {"combo": r["combo"], "params": r["params"], "task": t,
                   "type": m["type"]}
            if m["type"] == "regression":
                row.update({k: m[k] for k in ["mae", "rmse", "r2", "corr"]})
            else:
                row.update({k: m[k] for k in ["accuracy", "f1_weighted", "f1_macro"]})
            rows_flat.append(row)

csv_path = os.path.join(BASE_SAVE_DIR, "multitask_mixed_summary.csv")
pd.DataFrame(rows_flat).to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")